In [1]:
from pathlib import Path
import os

root = Path.cwd()

while not (root / ".git").exists():
    root = root.parent

os.chdir(root)

print("Project root:", root)

Project root: d:\Projects\btc-research-lab


In [3]:
import nbformat
from pathlib import Path

nb_path = Path("notebooks/phase3_H02-H04c.ipynb")

nb = nbformat.read(nb_path, as_version=4)

keywords = [
    "vol_regime",
    "high_vol",
    "quantile",
    "qcut",
    "DDD",
    "strongest",
    "pattern"
]

for i, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue

    src = cell.source.lower()

    if any(k.lower() in src for k in keywords):
        print("\n" + "="*80)
        print(f"CELL {i}")
        print("="*80)
        print(cell.source[:4000])  # فقط 4000 کاراکتر اول


CELL 2
work_df = df.copy()

# H-02 : Compression -> Expansion

work_df['ret'] = work_df['close'].pct_change()

work_df['rolling_vol_48'] = (
    work_df['ret']
    .rolling(48)
    .std()
)

vol_threshold = work_df['rolling_vol_48'].quantile(0.10)

work_df['compression'] = (
    work_df['rolling_vol_48'] <= vol_threshold
)

work_df['fwd_6'] = work_df['close'].shift(-6)/work_df['close'] - 1
work_df['fwd_12'] = work_df['close'].shift(-12)/work_df['close'] - 1
work_df['fwd_24'] = work_df['close'].shift(-24)/work_df['close'] - 1

expansion_threshold = 0.005  # 0.5%

subset = work_df.loc[work_df['compression']].copy()

subset['big_move_12'] = (
    subset['fwd_12'].abs() > expansion_threshold
)

baseline_prob = (
    work_df['fwd_12'].abs() > expansion_threshold
).mean()

event_prob = subset['big_move_12'].mean()

lift = event_prob / baseline_prob

print("="*50)
print("H-02 Compression -> Expansion")
print("="*50)
print("n =", len(subset))
print("baseline_prob =", round(baseline_prob,4))
p

In [4]:
import nbformat
from pathlib import Path

nb = nbformat.read(
    "notebooks/phase3_H02-H04c.ipynb",
    as_version=4
)

for i, cell in enumerate(nb.cells):

    if cell.cell_type != "code":
        continue

    src = cell.source.lower()

    if (
        "ddd" in src
        or "vol_regime" in src
        or "qcut" in src
        or "quantile" in src
        or "strongest" in src
        or "top10" in src
        or "h04" in src
    ):
        print("\n" + "="*80)
        print(f"CELL {i}")
        print("="*80)
        print(cell.source)


CELL 2
work_df = df.copy()

# H-02 : Compression -> Expansion

work_df['ret'] = work_df['close'].pct_change()

work_df['rolling_vol_48'] = (
    work_df['ret']
    .rolling(48)
    .std()
)

vol_threshold = work_df['rolling_vol_48'].quantile(0.10)

work_df['compression'] = (
    work_df['rolling_vol_48'] <= vol_threshold
)

work_df['fwd_6'] = work_df['close'].shift(-6)/work_df['close'] - 1
work_df['fwd_12'] = work_df['close'].shift(-12)/work_df['close'] - 1
work_df['fwd_24'] = work_df['close'].shift(-24)/work_df['close'] - 1

expansion_threshold = 0.005  # 0.5%

subset = work_df.loc[work_df['compression']].copy()

subset['big_move_12'] = (
    subset['fwd_12'].abs() > expansion_threshold
)

baseline_prob = (
    work_df['fwd_12'].abs() > expansion_threshold
).mean()

event_prob = subset['big_move_12'].mean()

lift = event_prob / baseline_prob

print("="*50)
print("H-02 Compression -> Expansion")
print("="*50)
print("n =", len(subset))
print("baseline_prob =", round(baseline_prob,4))
p

In [6]:
from pathlib import Path

for ext in ["*.parquet", "*.csv"]:
    print(f"\n===== {ext} =====")

    files = sorted(Path(".").rglob(ext))

    for f in files:
        print(f)


===== *.parquet =====
.venv\Lib\site-packages\pyarrow\tests\data\parquet\v0.7.1.all-named-index.parquet
.venv\Lib\site-packages\pyarrow\tests\data\parquet\v0.7.1.column-metadata-handling.parquet
.venv\Lib\site-packages\pyarrow\tests\data\parquet\v0.7.1.parquet
.venv\Lib\site-packages\pyarrow\tests\data\parquet\v0.7.1.some-named-index.parquet
data\backtests\h04c_trades.parquet
data\processed\BTCUSDT_5m.parquet
data\processed\BTCUSDT_5m_regularized.parquet

===== *.csv =====
.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\data_x_x2_x3.csv
.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\msft.csv
.venv\Lib\site-packages\matplotlib\mpl-data\sample_data\Stocks.csv
.venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arccos.csv
.venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arccosh.csv
.venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arcsin.csv
.venv\Lib\site-packages\numpy\_core\tests\data\umath-validation-set-arcsinh.cs

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# Load base data
# -----------------------------

work_df = pd.read_parquet(
    "data/processed/BTCUSDT_5m.parquet"
).copy()

# -----------------------------
# H04c Signal Definition
# -----------------------------

work_df["ret_h04c"] = work_df["close"].pct_change()

work_df["sign_h04c"] = np.where(
    work_df["ret_h04c"] > 0,
    "U",
    "D"
)

work_df["pattern_h04c"] = (
      work_df["sign_h04c"].shift(2)
    + work_df["sign_h04c"].shift(1)
    + work_df["sign_h04c"]
)

work_df["ddd_strength_h04c"] = (
      work_df["ret_h04c"]
    + work_df["ret_h04c"].shift(1)
    + work_df["ret_h04c"].shift(2)
)

ddd_sample = work_df[
    work_df["pattern_h04c"] == "DDD"
].copy()

q10_threshold = (
    ddd_sample["ddd_strength_h04c"]
    .quantile(0.10)
)

work_df["signal_h04c"] = (
    (work_df["pattern_h04c"] == "DDD")
    &
    (work_df["ddd_strength_h04c"] <= q10_threshold)
)

# -----------------------------
# Signal Artifact
# -----------------------------

signal_df = work_df.loc[
    work_df["signal_h04c"],
    [
        "close",
        "pattern_h04c",
        "ddd_strength_h04c"
    ]
].copy()

signal_df["signal"] = "LONG"
signal_df["q10_threshold"] = q10_threshold

Path("data/signals").mkdir(
    parents=True,
    exist_ok=True
)

signal_df.to_parquet(
    "data/signals/h04c_signals.parquet"
)

# -----------------------------
# Summary
# -----------------------------

print("Rows in source data :", len(work_df))
print("DDD events          :", len(ddd_sample))
print("Signal count        :", len(signal_df))
print("Q10 threshold       :", round(q10_threshold, 6))

print("\nSaved:")
print("data/signals/h04c_signals.parquet")

print("\nPreview:")
print(signal_df.head())


Rows in source data : 630482
DDD events          : 69892
Signal count        : 6990
Q10 threshold       : -0.007527

Saved:
data/signals/h04c_signals.parquet

Preview:
        close pattern_h04c  ddd_strength_h04c signal  q10_threshold
1040  3712.36          DDD          -0.008712   LONG      -0.007527
1839  3966.05          DDD          -0.010731   LONG      -0.007527
1846  3934.13          DDD          -0.008823   LONG      -0.007527
2207  4000.48          DDD          -0.010939   LONG      -0.007527
2208  3987.24          DDD          -0.011759   LONG      -0.007527


In [8]:
from pathlib import Path
import pandas as pd

signals = pd.read_parquet(
    "data/signals/h04c_signals.parquet"
).copy()

signals = signals.sort_index()

hold_bars = 36

selected_idx = []
next_allowed = -1

for idx in signals.index:

    if idx >= next_allowed:
        selected_idx.append(idx)
        next_allowed = idx + hold_bars

exec_signals = signals.loc[selected_idx].copy()

Path("data/signals").mkdir(
    parents=True,
    exist_ok=True
)

exec_signals.to_parquet(
    "data/signals/h04c_executable_signals.parquet"
)

print("Raw signals       :", len(signals))
print("Executable trades :", len(exec_signals))
print("Reduction ratio   :", round(
    len(exec_signals) / len(signals),
    3
))

print("\nSaved:")
print("data/signals/h04c_executable_signals.parquet")

Raw signals       : 6990
Executable trades : 2839
Reduction ratio   : 0.406

Saved:
data/signals/h04c_executable_signals.parquet


In [9]:
from pathlib import Path
from datetime import datetime

# --------------------------------------------------
# Directories
# --------------------------------------------------

Path("reports").mkdir(exist_ok=True)
Path("data/signals").mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# H04c Specification
# --------------------------------------------------

spec_text = f"""
# H04c Specification v1.0

Generated:
{datetime.utcnow().isoformat()} UTC

--------------------------------------------------

Pattern

DDD

Definition

ret[t]   < 0
ret[t-1] < 0
ret[t-2] < 0

--------------------------------------------------

Strength

ddd_strength =
ret[t]
+ ret[t-1]
+ ret[t-2]

Threshold

Q10 of DDD sample

Observed Threshold

{q10_threshold:.6f}

--------------------------------------------------

Direction

Long Only

--------------------------------------------------

Holding Period

36 bars

--------------------------------------------------

Transaction Cost

4 bp round-trip

--------------------------------------------------

Position Overlap

Not Allowed

--------------------------------------------------

Stop Loss

None

--------------------------------------------------

Leverage

None

--------------------------------------------------

Phase 5 Reconstruction Check

Raw Signals      : {6990}
Executable Trades: {2839}

Result:

Executable trade count matches
historical backtest (~2840 trades)

--------------------------------------------------

Status

PASSED RECONSTRUCTION

Ready for Deployment Research

--------------------------------------------------
"""

with open(
    "reports/H04c_specification.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(spec_text)

print("Saved:")
print("reports/H04c_specification.md")
print("data/signals/h04c_signals.parquet")
print("data/signals/h04c_executable_signals.parquet")

Saved:
reports/H04c_specification.md
data/signals/h04c_signals.parquet
data/signals/h04c_executable_signals.parquet


C:\Users\bande.HEKMAT\AppData\Local\Temp\ipykernel_2612\3502703929.py:19: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {datetime.utcnow().isoformat()} UTC


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------
# Load
# -----------------------------

df = pd.read_parquet(
    "data/processed/BTCUSDT_5m.parquet"
)

signals = pd.read_parquet(
    "data/signals/h04c_executable_signals.parquet"
)

# -----------------------------
# Rebuild Trades
# -----------------------------

hold_bars = 36
cost = 0.0004

trades = []

for idx in signals.index:

    exit_idx = idx + hold_bars

    if exit_idx >= len(df):
        continue

    entry_price = df.loc[idx, "close"]
    exit_price = df.loc[exit_idx, "close"]

    ret = (
        exit_price / entry_price
        - 1
        - cost
    )

    trades.append(ret)

trades = pd.Series(trades)

# -----------------------------
# Metrics
# -----------------------------

equity = (1 + trades).cumprod()

gross_profit = trades[trades > 0].sum()
gross_loss = abs(trades[trades < 0].sum())

pf = gross_profit / gross_loss

hit_rate = (trades > 0).mean()

dd = (
    equity /
    equity.cummax()
    - 1
)

print("Trades      :", len(trades))
print("Mean Return :", round(trades.mean(),6))
print("Hit Rate    :", round(hit_rate,4))
print("PF          :", round(pf,3))
print("Max DD      :", round(dd.min(),3))
print("Final Eq    :", round(equity.iloc[-1],3))

Trades      : 2839
Mean Return : 0.000483
Hit Rate    : 0.5498
PF          : 1.08
Max DD      : -0.585
Final Eq    : 2.321


In [11]:
import nbformat

nb = nbformat.read(
    "notebooks/phase3_H02-H04c.ipynb",
    as_version=4
)

for i, cell in enumerate(nb.cells):

    if cell.cell_type != "code":
        continue

    src = cell.source.lower()

    if (
        "hold" in src
        or "36" in src
        or "equity" in src
        or "profit factor" in src
        or "pf" in src
        or "trade_return" in src
        or "entry_price" in src
        or "exit_price" in src
        or "cost" in src
        or "backtester" in src
    ):
        print("\n" + "="*80)
        print(f"CELL {i}")
        print("="*80)
        print(cell.source)


CELL 2
work_df = df.copy()

# H-02 : Compression -> Expansion

work_df['ret'] = work_df['close'].pct_change()

work_df['rolling_vol_48'] = (
    work_df['ret']
    .rolling(48)
    .std()
)

vol_threshold = work_df['rolling_vol_48'].quantile(0.10)

work_df['compression'] = (
    work_df['rolling_vol_48'] <= vol_threshold
)

work_df['fwd_6'] = work_df['close'].shift(-6)/work_df['close'] - 1
work_df['fwd_12'] = work_df['close'].shift(-12)/work_df['close'] - 1
work_df['fwd_24'] = work_df['close'].shift(-24)/work_df['close'] - 1

expansion_threshold = 0.005  # 0.5%

subset = work_df.loc[work_df['compression']].copy()

subset['big_move_12'] = (
    subset['fwd_12'].abs() > expansion_threshold
)

baseline_prob = (
    work_df['fwd_12'].abs() > expansion_threshold
).mean()

event_prob = subset['big_move_12'].mean()

lift = event_prob / baseline_prob

print("="*50)
print("H-02 Compression -> Expansion")
print("="*50)
print("n =", len(subset))
print("baseline_prob =", round(baseline_prob,4))
p

In [12]:
from pathlib import Path
import pandas as pd

for f in Path("data").rglob("*"):

    if f.suffix in [".parquet", ".csv"]:
        print(f)

data\backtests\h04c_trades.parquet
data\processed\BTCUSDT_5m.parquet
data\processed\BTCUSDT_5m_regularized.parquet
data\signals\h04c_executable_signals.parquet
data\signals\h04c_signals.parquet
data\validation\phase4_position_sizing.csv
data\validation\phase4_walkforward.csv


In [13]:
import pandas as pd

trades = pd.read_parquet(
    "data/backtests/h04c_trades.parquet"
)

print("Rows:", len(trades))
print()

print("Columns:")
print(trades.columns.tolist())
print()

print(trades.head())
print()

if "net_ret" in trades.columns:

    tr = trades["net_ret"]

    pf = (
        tr[tr > 0].sum()
        /
        abs(tr[tr <= 0].sum())
    )

    equity = (1 + tr).cumprod()

    dd = (
        equity
        /
        equity.cummax()
        - 1
    )

    print("PF      :", round(pf,3))
    print("Max DD  :", round(dd.min(),3))
    print("Equity  :", round(equity.iloc[-1],3))

Rows: 2840

Columns:
['entry_time', 'exit_time', 'entry_price', 'exit_price', 'gross_ret', 'net_ret', 'duration', 'year', 'equity', 'vol_regime', 'bull', 'mae', 'mfe']

           entry_time           exit_time  entry_price  exit_price  gross_ret  \
0 2019-01-04 14:40:00 2019-01-04 17:40:00      3712.36     3738.72   0.007101   
1 2019-01-07 09:15:00 2019-01-07 12:15:00      3966.05     3974.00   0.002005   
2 2019-01-08 15:55:00 2019-01-08 18:55:00      4000.48     3968.46  -0.008004   
3 2019-01-10 06:30:00 2019-01-10 09:30:00      3812.86     3743.00  -0.018322   
4 2019-01-10 16:10:00 2019-01-10 19:10:00      3543.99     3563.87   0.005609   

    net_ret  duration  year    equity vol_regime   bull       mae       mfe  
0  0.006701        36  2019  1.006701        Low  False -0.002368  0.011217  
1  0.001605        36  2019  1.008316        Low  False -0.011225  0.007804  
2 -0.008404        36  2019  0.999842        Mid  False -0.015413  0.005262  
3 -0.018722        36  2019  0.9

In [14]:
from pathlib import Path
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "data/processed/BTCUSDT_5m_regularized.parquet"
)

signals = pd.read_parquet(
    "data/signals/h04c_signals.parquet"
)

work_df = df.copy()

work_df["ret"] = work_df["close"].pct_change()

work_df["sign"] = np.where(
    work_df["ret"] > 0,
    "U",
    "D"
)

work_df["pattern_h04c"] = (
    work_df["sign"].shift(2)
    + work_df["sign"].shift(1)
    + work_df["sign"]
)

work_df["ddd_strength_h04c"] = (
      work_df["ret"]
    + work_df["ret"].shift(1)
    + work_df["ret"].shift(2)
)

threshold = (
    work_df.loc[
        work_df["pattern_h04c"] == "DDD",
        "ddd_strength_h04c"
    ]
    .quantile(0.10)
)

work_df["signal"] = (
    (work_df["pattern_h04c"] == "DDD")
    &
    (work_df["ddd_strength_h04c"] <= threshold)
)

reconstructed = work_df["signal"].sum()

saved = len(signals)

print("Reconstructed :", reconstructed)
print("Saved         :", saved)
print("Match         :", reconstructed == saved)

Reconstructed : 7070
Saved         : 6990
Match         : False


In [15]:
import pandas as pd

saved = pd.read_parquet(
    "data/signals/h04c_signals.parquet"
)

df = pd.read_parquet(
    "data/processed/BTCUSDT_5m_regularized.parquet"
)

work_df = df.copy()

work_df["ret"] = work_df["close"].pct_change()

work_df["sign"] = (work_df["ret"] > 0).map(
    {True: "U", False: "D"}
)

work_df["pattern_h04c"] = (
    work_df["sign"].shift(2)
    + work_df["sign"].shift(1)
    + work_df["sign"]
)

work_df["ddd_strength_h04c"] = (
      work_df["ret"]
    + work_df["ret"].shift(1)
    + work_df["ret"].shift(2)
)

threshold = (
    work_df.loc[
        work_df["pattern_h04c"] == "DDD",
        "ddd_strength_h04c"
    ].quantile(0.10)
)

reconstructed = work_df[
    (work_df["pattern_h04c"] == "DDD")
    &
    (work_df["ddd_strength_h04c"] <= threshold)
].copy()

saved_idx = set(saved.index)
recon_idx = set(reconstructed.index)

print("Saved only :", len(saved_idx - recon_idx))
print("Recon only :", len(recon_idx - saved_idx))
print("Intersection :", len(saved_idx & recon_idx))

Saved only : 6774
Recon only : 6854
Intersection : 216


In [16]:
import pandas as pd

saved = pd.read_parquet(
    "data/signals/h04c_signals.parquet"
)

print(saved.head())

print()
print(type(saved.index))

print()
print(saved.index[:10])

print()
print(saved.columns.tolist())

        close pattern_h04c  ddd_strength_h04c signal  q10_threshold
1040  3712.36          DDD          -0.008712   LONG      -0.007527
1839  3966.05          DDD          -0.010731   LONG      -0.007527
1846  3934.13          DDD          -0.008823   LONG      -0.007527
2207  4000.48          DDD          -0.010939   LONG      -0.007527
2208  3987.24          DDD          -0.011759   LONG      -0.007527

<class 'pandas.Index'>

Index([1040, 1839, 1846, 2207, 2208, 2670, 2671, 2786, 3650, 4804], dtype='int64')

['close', 'pattern_h04c', 'ddd_strength_h04c', 'signal', 'q10_threshold']


In [17]:
print(saved.index.min())
print(saved.index.max())

1040
630405


In [18]:
from pathlib import Path
import pandas as pd

root = Path.cwd()

while not (root / ".git").exists():
    root = root.parent

df = pd.read_parquet(
    root / "data" / "processed" / "BTCUSDT_5m_regularized.parquet"
)

signals = pd.read_parquet(
    root / "data" / "signals" / "h04c_signals.parquet"
)

signals_v2 = signals.copy()

signals_v2["timestamp"] = (
    df.loc[signals_v2.index].index
)

signals_v2 = signals_v2[
    [
        "timestamp",
        "close",
        "pattern_h04c",
        "ddd_strength_h04c",
        "signal",
        "q10_threshold"
    ]
]

signals_v2.to_parquet(
    root / "data" / "signals" / "h04c_signals.parquet",
    index=False
)

print(signals_v2.head())
print()
print("Saved upgraded signal file.")

      timestamp    close pattern_h04c  ddd_strength_h04c signal  q10_threshold
1040       1040  3712.36          DDD          -0.008712   LONG      -0.007527
1839       1839  3966.05          DDD          -0.010731   LONG      -0.007527
1846       1846  3934.13          DDD          -0.008823   LONG      -0.007527
2207       2207  4000.48          DDD          -0.010939   LONG      -0.007527
2208       2208  3987.24          DDD          -0.011759   LONG      -0.007527

Saved upgraded signal file.


In [19]:
print(df.head())

print()
print(df.columns.tolist())

print()
print(type(df.index))

print()
print(df.index[:5])

            open_time     open     high      low    close     volume
0 2019-01-01 00:00:00  3701.23  3703.72  3695.00  3696.32  85.572181
1 2019-01-01 00:05:00  3696.30  3697.24  3689.88  3692.34  62.296581
2 2019-01-01 00:10:00  3692.34  3698.93  3692.34  3697.31  43.105333
3 2019-01-01 00:15:00  3697.91  3698.75  3693.00  3693.00  48.551084
4 2019-01-01 00:20:00  3693.44  3695.98  3690.92  3692.18  47.706443

['open_time', 'open', 'high', 'low', 'close', 'volume']

<class 'pandas.RangeIndex'>

RangeIndex(start=0, stop=5, step=1)


In [20]:
from pathlib import Path
import pandas as pd

root = Path.cwd()

while not (root / ".git").exists():
    root = root.parent

df = pd.read_parquet(
    root / "data" / "processed" / "BTCUSDT_5m_regularized.parquet"
)

signals = pd.read_parquet(
    root / "data" / "signals" / "h04c_signals.parquet"
)

signals_v2 = signals.copy()

signals_v2["timestamp"] = (
    df.loc[signals_v2.index, "open_time"].values
)

signals_v2 = signals_v2[
    [
        "timestamp",
        "close",
        "pattern_h04c",
        "ddd_strength_h04c",
        "signal",
        "q10_threshold"
    ]
]

signals_v2.to_parquet(
    root / "data" / "signals" / "h04c_signals.parquet",
    index=False
)

print(signals_v2.head())
print()
print("Rows:", len(signals_v2))
print("Saved.")

            timestamp    close pattern_h04c  ddd_strength_h04c signal  \
0 2019-01-01 00:00:00  3712.36          DDD          -0.008712   LONG   
1 2019-01-01 00:05:00  3966.05          DDD          -0.010731   LONG   
2 2019-01-01 00:10:00  3934.13          DDD          -0.008823   LONG   
3 2019-01-01 00:15:00  4000.48          DDD          -0.010939   LONG   
4 2019-01-01 00:20:00  3987.24          DDD          -0.011759   LONG   

   q10_threshold  
0      -0.007527  
1      -0.007527  
2      -0.007527  
3      -0.007527  
4      -0.007527  

Rows: 6990
Saved.


In [21]:
import pandas as pd

sig = pd.read_parquet(
    "data/signals/h04c_executable_signals.parquet"
)

print(sig.head())

print()
print(type(sig.index))
print(sig.index[:10])

print()
print(sig.columns.tolist())

        close pattern_h04c  ddd_strength_h04c signal  q10_threshold
1040  3712.36          DDD          -0.008712   LONG      -0.007527
1839  3966.05          DDD          -0.010731   LONG      -0.007527
2207  4000.48          DDD          -0.010939   LONG      -0.007527
2670  3812.86          DDD          -0.041164   LONG      -0.007527
2786  3543.99          DDD          -0.049616   LONG      -0.007527

<class 'pandas.Index'>
Index([1040, 1839, 2207, 2670, 2786, 3650, 4804, 5610, 6510, 7849], dtype='int64')

['close', 'pattern_h04c', 'ddd_strength_h04c', 'signal', 'q10_threshold']


In [22]:
from pathlib import Path
import pandas as pd

root = Path.cwd()

while not (root / ".git").exists():
    root = root.parent

df = pd.read_parquet(
    root / "data" / "processed" / "BTCUSDT_5m_regularized.parquet"
)

exec_sig = pd.read_parquet(
    root / "data" / "signals" / "h04c_executable_signals.parquet"
)

signals = exec_sig.copy()

signals["timestamp"] = (
    df.loc[signals.index, "open_time"].values
)

signals = signals[
    [
        "timestamp",
        "close",
        "pattern_h04c",
        "ddd_strength_h04c",
        "signal",
        "q10_threshold"
    ]
]

signals.to_parquet(
    root / "data" / "signals" / "h04c_signals.parquet",
    index=False
)

print(signals.head())
print()
print("Rows:", len(signals))
print("Saved corrected h04c_signals.parquet")

               timestamp    close pattern_h04c  ddd_strength_h04c signal  \
1040 2019-01-04 14:40:00  3712.36          DDD          -0.008712   LONG   
1839 2019-01-07 09:15:00  3966.05          DDD          -0.010731   LONG   
2207 2019-01-08 15:55:00  4000.48          DDD          -0.010939   LONG   
2670 2019-01-10 06:30:00  3812.86          DDD          -0.041164   LONG   
2786 2019-01-10 16:10:00  3543.99          DDD          -0.049616   LONG   

      q10_threshold  
1040      -0.007527  
1839      -0.007527  
2207      -0.007527  
2670      -0.007527  
2786      -0.007527  

Rows: 2839
Saved corrected h04c_signals.parquet


In [23]:
from pathlib import Path

base = Path(".")

paths = {
    "signals": base / "data" / "signals",
    "backtests": base / "data" / "backtests",
    "features": base / "data" / "features",
    "reports": base / "reports",
}

for name, p in paths.items():
    print(name, "exists:", p.exists())
    if p.exists():
        print("  files:", len(list(p.glob("*"))))

signals exists: True
  files: 2
backtests exists: True
  files: 1
features exists: True
  files: 0
reports exists: True
  files: 6


In [24]:
from pathlib import Path
import pandas as pd

base = Path(".")

dirs = {
    "signals": base / "data" / "signals",
    "backtests": base / "data" / "backtests",
    "features": base / "data" / "features",
    "reports": base / "reports",
}

print("=== FILE INVENTORY ===")
for name, path in dirs.items():
    print(f"\n[{name}]")
    if path.exists():
        files = list(path.glob("*"))
        print("count:", len(files))
        for f in files:
            print(" -", f.name)
    else:
        print("missing directory")

print("\n=== SAMPLE SIGNAL FILE INSPECTION ===")
sig_files = list(dirs["signals"].glob("*"))
if sig_files:
    df_sig = pd.read_parquet(sig_files[0])
    print("signal file:", sig_files[0].name)
    print("columns:", df_sig.columns.tolist())
    print("shape:", df_sig.shape)
    print(df_sig.head(3))

print("\n=== SAMPLE BACKTEST FILE INSPECTION ===")
bt_files = list(dirs["backtests"].glob("*"))
if bt_files:
    df_bt = pd.read_parquet(bt_files[0])
    print("backtest file:", bt_files[0].name)
    print("columns:", df_bt.columns.tolist())
    print("shape:", df_bt.shape)
    print(df_bt.head(3))

=== FILE INVENTORY ===

[signals]
count: 2
 - h04c_executable_signals.parquet
 - h04c_signals.parquet

[backtests]
count: 1
 - h04c_trades.parquet

[features]
count: 0

[reports]
count: 6
 - H04c_specification.md
 - Phase 2
 - phase1 Summary.md
 - phase2_combined.md
 - phase3_h04c_summary.md
 - phase4_validation_report.md

=== SAMPLE SIGNAL FILE INSPECTION ===
signal file: h04c_executable_signals.parquet
columns: ['close', 'pattern_h04c', 'ddd_strength_h04c', 'signal', 'q10_threshold']
shape: (2839, 5)
        close pattern_h04c  ddd_strength_h04c signal  q10_threshold
1040  3712.36          DDD          -0.008712   LONG      -0.007527
1839  3966.05          DDD          -0.010731   LONG      -0.007527
2207  4000.48          DDD          -0.010939   LONG      -0.007527

=== SAMPLE BACKTEST FILE INSPECTION ===
backtest file: h04c_trades.parquet
columns: ['entry_time', 'exit_time', 'entry_price', 'exit_price', 'gross_ret', 'net_ret', 'duration', 'year', 'equity', 'vol_regime', 'bull', 'm

In [25]:
import pandas as pd
from pathlib import Path

sig_path = Path("data/signals/h04c_executable_signals.parquet")
df = pd.read_parquet(sig_path)

work_df = df.copy()

# simple execution model assumptions
slippage_bps = 2  # very conservative assumption
fee_bps = 4

work_df["side"] = work_df["signal"]

# simulate fill price
work_df["fill_price"] = work_df["close"] * (1 + (slippage_bps / 10000) * 
                                            work_df["signal"].map({"LONG": 1, "SHORT": -1}))

# dummy pnl proxy (not final)
work_df["exec_return_proxy"] = (
    -work_df["signal"].map({"LONG": -1, "SHORT": 1}) * 0.001  # placeholder micro edge shift
)

print("EXECUTION LAYER CREATED")
print(work_df[["close", "signal", "fill_price", "exec_return_proxy"]].head())

print("\nSignal count:", len(work_df))
print("Long/Short split:")
print(work_df["signal"].value_counts())

EXECUTION LAYER CREATED
        close signal   fill_price  exec_return_proxy
1040  3712.36   LONG  3713.102472              0.001
1839  3966.05   LONG  3966.843210              0.001
2207  4000.48   LONG  4001.280096              0.001
2670  3812.86   LONG  3813.622572              0.001
2786  3543.99   LONG  3544.698798              0.001

Signal count: 2839
Long/Short split:
signal
LONG    2839
Name: count, dtype: int64


In [26]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")

work_df = df.copy()

# --- regime mapping (approx from report assumption)
def regime_multiplier(x):
    return 1.3 if "High" in str(x) else 0.0

# --- signal strength proxy
work_df["strength"] = np.abs(work_df["ddd_strength_h04c"])

# normalize strength
work_df["strength_norm"] = work_df["strength"] / work_df["strength"].max()

# Kelly baseline from Phase 4
kelly = 0.0243

# sizing function
work_df["position_size"] = kelly * work_df["strength_norm"]

# regime placeholder (since not fully defined here)
work_df["regime_multiplier"] = 1.3  # temporary assumption

work_df["final_size"] = work_df["position_size"] * work_df["regime_multiplier"]

print("=== SIZE STATS ===")
print(work_df["final_size"].describe())

print("\n=== SIGNAL WEIGHTING EFFECT ===")
print(work_df[["close","signal","final_size","strength_norm"]].head())

=== SIZE STATS ===
count    2839.000000
mean        0.004521
std         0.002208
min         0.002994
25%         0.003322
50%         0.003822
75%         0.004797
max         0.031590
Name: final_size, dtype: float64

=== SIGNAL WEIGHTING EFFECT ===
        close signal  final_size  strength_norm
1040  3712.36   LONG    0.003466       0.109712
1839  3966.05   LONG    0.004269       0.135135
2207  4000.48   LONG    0.004352       0.137750
2670  3812.86   LONG    0.016375       0.518371
2786  3543.99   LONG    0.019737       0.624800


In [27]:
import pandas as pd
import numpy as np

df_sig = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
df_bt = pd.read_parquet("data/backtests/h04c_trades.parquet")

work_df = df_sig.copy()

# merge signals with realized returns
df_bt_small = df_bt[["entry_price", "net_ret"]].copy()

work_df = work_df.merge(df_bt_small, left_index=True, right_index=True, how="inner")

# bucket strength
work_df["strength_bucket"] = pd.qcut(work_df["ddd_strength_h04c"], 5, labels=False)

print("=== EXPECTANCY BY STRENGTH ===")
print(work_df.groupby("strength_bucket")["net_ret"].agg(["mean","count","std"]))

print("\n=== EXPECTANCY BY SIGNAL TYPE ===")
print(work_df.groupby("signal")["net_ret"].mean())

print("\n=== SKEW CHECK ===")
print(work_df.groupby("strength_bucket")["net_ret"].mean())

=== EXPECTANCY BY STRENGTH ===
                     mean  count  std
strength_bucket                      
0                0.008929      1  NaN
1                0.018265      1  NaN
2                0.001908      1  NaN
3                0.010207      1  NaN
4                0.013237      1  NaN

=== EXPECTANCY BY SIGNAL TYPE ===
signal
LONG    0.010509
Name: net_ret, dtype: float64

=== SKEW CHECK ===
strength_bucket
0    0.008929
1    0.018265
2    0.001908
3    0.010207
4    0.013237
Name: net_ret, dtype: float64


In [28]:
import pandas as pd
import hashlib

sig = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
bt = pd.read_parquet("data/backtests/h04c_trades.parquet")

work_sig = sig.copy()
work_bt = bt.copy()

# create event ID based on index (temporary robust key)
work_sig = work_sig.reset_index().rename(columns={"index":"event_id"})
work_bt = work_bt.reset_index().rename(columns={"index":"event_id"})

# check alignment sanity
print("SIGNAL IDS:", work_sig["event_id"].head())
print("TRADE IDS:", work_bt["event_id"].head())

# merge properly
merged = work_sig.merge(work_bt[["event_id","net_ret"]], on="event_id", how="inner")

print("\n=== VALID MERGED SAMPLE SIZE ===")
print(len(merged))

print("\n=== EXPECTANCY CHECK (CORRECTED) ===")
merged["bucket"] = pd.qcut(merged["ddd_strength_h04c"], 5, duplicates="drop")
print(merged.groupby("bucket")["net_ret"].agg(["mean","count"]))

SIGNAL IDS: 0    1040
1    1839
2    2207
3    2670
4    2786
Name: event_id, dtype: int64
TRADE IDS: 0    0
1    1
2    2
3    3
4    4
Name: event_id, dtype: int64

=== VALID MERGED SAMPLE SIZE ===
5

=== EXPECTANCY CHECK (CORRECTED) ===
                         mean  count
bucket                              
(-0.0506, -0.0429]   0.008929      1
(-0.0429, -0.023]    0.018265      1
(-0.023, -0.0109]    0.001908      1
(-0.0109, -0.0103]   0.010207      1
(-0.0103, -0.00871]  0.013237      1


In [29]:
import pandas as pd

sig = pd.read_parquet("data/signals/h04c_executable_signals.parquet")

work = sig.copy().sort_values("close").reset_index(drop=True)

# causal trade construction (simple fixed-horizon model)
H = 36

trades = []

for i in range(len(work)-H):
    if work.loc[i, "signal"] != "LONG":
        continue

    entry_price = work.loc[i, "close"]
    exit_price = work.loc[i+H, "close"]

    net_ret = (exit_price - entry_price) / entry_price

    trades.append({
        "entry_idx": i,
        "exit_idx": i+H,
        "entry_price": entry_price,
        "exit_price": exit_price,
        "net_ret": net_ret,
        "strength": work.loc[i, "ddd_strength_h04c"]
    })

trades_df = pd.DataFrame(trades)

print("=== CAUSAL TRADES ===")
print(len(trades_df))

print("\n=== REAL EXPECTANCY BY STRENGTH ===")
trades_df["bucket"] = pd.qcut(trades_df["strength"], 5, duplicates="drop")
print(trades_df.groupby("bucket")["net_ret"].agg(["mean","count"]))

=== CAUSAL TRADES ===
2803

=== REAL EXPECTANCY BY STRENGTH ===
                          mean  count
bucket                               
(-0.0804, -0.013]     0.048998    561
(-0.013, -0.0104]     0.044066    560
(-0.0104, -0.00905]   0.041136    561
(-0.00905, -0.00815]  0.040381    560
(-0.00815, -0.00753]  0.044566    561


In [30]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")

work = df.copy().reset_index(drop=True)

H = 36

trades = []

for i in range(len(work)-H):

    if work.loc[i, "signal"] != "LONG":
        continue

    entry = work.loc[i+1, "close"]  # next-bar execution (more realistic)
    exit_ = work.loc[i+H, "close"]

    ret = (exit_ - entry) / entry

    # simple slippage model (regime proxy via volatility of strength)
    slip = 0.0005 + 0.002 * abs(work.loc[i, "ddd_strength_h04c"])

    net_ret = ret - slip

    trades.append({
        "strength": work.loc[i, "ddd_strength_h04c"],
        "net_ret": net_ret
    })

trades_df = pd.DataFrame(trades)

print("=== MICROSTRUCTURE ADJUSTED EXPECTANCY ===")
print(trades_df["net_ret"].mean())

print("\n=== BY STRENGTH QUINTILE ===")
trades_df["bucket"] = pd.qcut(trades_df["strength"], 5, duplicates="drop")
print(trades_df.groupby("bucket")["net_ret"].mean())

=== MICROSTRUCTURE ADJUSTED EXPECTANCY ===
0.05971096173819001

=== BY STRENGTH QUINTILE ===
bucket
(-0.0804, -0.013]       0.063398
(-0.013, -0.0104]       0.058158
(-0.0104, -0.00905]     0.053508
(-0.00905, -0.00815]    0.057403
(-0.00815, -0.00753]    0.066081
Name: net_ret, dtype: float64


In [31]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

H = 36

trades = []

for i in range(len(work) - H):

    if work.loc[i, "signal"] != "LONG":
        continue

    strength = abs(work.loc[i, "ddd_strength_h04c"])

    entry_price = work.loc[i+1, "close"]

    exit_price = work.loc[i+H, "close"]

    # volatility proxy
    vol = work["close"].pct_change().rolling(50).std().iloc[i]

    if np.isnan(vol):
        vol = 0.001

    # slippage model
    slippage = 0.0003 + 0.5 * vol + 0.0008 * strength

    gross_ret = (exit_price - entry_price) / entry_price

    net_ret = gross_ret - slippage - 0.0004  # fee

    trades.append({
        "net_ret": net_ret,
        "strength": strength,
        "vol": vol
    })

trades_df = pd.DataFrame(trades)

print("=== EXECUTION ENGINE v6 RESULTS ===")
print(trades_df["net_ret"].mean())

print("\n=== STRENGTH EFFECT ===")
print(trades_df.groupby(pd.qcut(trades_df["strength"], 5, duplicates="drop"))["net_ret"].mean())

print("\n=== VOL EFFECT ===")
print(trades_df["net_ret"].corr(trades_df["vol"]))

=== EXECUTION ENGINE v6 RESULTS ===
0.044922281085780565

=== STRENGTH EFFECT ===
strength
(0.00653, 0.00815]    0.051609
(0.00815, 0.00905]    0.042735
(0.00905, 0.0104]     0.038770
(0.0104, 0.013]       0.043412
(0.013, 0.0794]       0.048080
Name: net_ret, dtype: float64

=== VOL EFFECT ===
-0.15747992728744672


In [32]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.copy()

# volatility
work["ret"] = work["close"].pct_change()
work["vol"] = work["ret"].rolling(50).std()

# trend proxy
work["trend"] = work["close"].rolling(50).mean().pct_change()

# friction proxy (liquidity stress approximation)
work["friction"] = work["vol"] * abs(work["ddd_strength_h04c"])

# combined regime score
work["regime_score"] = work["vol"] * (1 + work["trend"]) * (1 + work["friction"])

print(work[["vol","trend","friction","regime_score"]].describe())

print("\n=== REGIME BUCKET EFFECT ===")
work["bucket"] = pd.qcut(work["regime_score"], 5, duplicates="drop")

# attach pseudo returns again (same execution logic)
H = 36
rets = []

for i in range(len(work)-H):
    if work.loc[i,"signal"] != "LONG":
        continue

    r = (work.loc[i+H,"close"] - work.loc[i+1,"close"]) / work.loc[i+1,"close"]
    rets.append((work.loc[i,"regime_score"], r))

tmp = pd.DataFrame(rets, columns=["regime","ret"])
tmp["bucket"] = pd.qcut(tmp["regime"], 5, duplicates="drop")

print(tmp.groupby("bucket")["ret"].mean())

               vol        trend     friction  regime_score
count  2789.000000  2789.000000  2789.000000   2789.000000
mean      0.029729     0.001133     0.000341      0.029777
std       0.004732     0.004587     0.000206      0.004762
min       0.018985    -0.014718     0.000157      0.018923
25%       0.026681    -0.001886     0.000241      0.026654
50%       0.029035     0.000753     0.000284      0.029093
75%       0.031852     0.004030     0.000367      0.031930
max       0.052735     0.015635     0.003484      0.052461

=== REGIME BUCKET EFFECT ===


KeyError: 0

In [33]:
for i in range(len(work)-H):
    row = work.iloc[i]
    next_row = work.iloc[i+1]
    exit_row = work.iloc[i+H]

    if row["signal"] != "LONG":
        continue

In [34]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

H = 36

trades = []

for i in range(len(work) - H):

    row = work.iloc[i]

    if row["signal"] != "LONG":
        continue

    entry = work.iloc[i+1]["close"]
    exit_ = work.iloc[i+H]["close"]

    ret = (exit_ - entry) / entry

    vol = work["close"].pct_change().rolling(50).std().iloc[i]
    if np.isnan(vol):
        vol = 0.001

    slip = 0.0003 + 0.5 * vol + 0.0008 * abs(row["ddd_strength_h04c"])

    net = ret - slip - 0.0004

    trades.append({
        "net_ret": net,
        "strength": row["ddd_strength_h04c"],
        "vol": vol
    })

trades_df = pd.DataFrame(trades)

print("=== FIXED ENGINE RESULTS ===")
print(trades_df["net_ret"].mean())

print("\n=== STRENGTH EFFECT ===")
print(trades_df.groupby(pd.qcut(trades_df["strength"], 5, duplicates="drop"))["net_ret"].mean())

print("\n=== VOL EFFECT ===")
print(trades_df["net_ret"].corr(trades_df["vol"]))

=== FIXED ENGINE RESULTS ===
0.044922281085780565

=== STRENGTH EFFECT ===
strength
(-0.0804, -0.013]       0.048080
(-0.013, -0.0104]       0.043412
(-0.0104, -0.00905]     0.038770
(-0.00905, -0.00815]    0.042735
(-0.00815, -0.00753]    0.051609
Name: net_ret, dtype: float64

=== VOL EFFECT ===
-0.15747992728744672


In [35]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")

work = df.reset_index(drop=True).copy()

# returns
work["ret"] = work["close"].pct_change()

# 1. volatility
work["vol"] = work["ret"].rolling(50).std()

# 2. instability (jumpiness)
work["instability"] = (work["ret"] - work["ret"].shift(1)).abs()

# 3. friction pressure
work["friction"] = work["vol"] * work["instability"] * work["ddd_strength_h04c"].abs()

# 4. final regime score
work["LARE"] = work["vol"] * (1 + work["instability"]) * (1 + work["friction"])

work = work.dropna()

print("=== LARE STATS ===")
print(work["LARE"].describe())

# attach simplified PnL model (same as before but consistent)
H = 36

trades = []

for i in range(len(work) - H):

    if work.iloc[i]["signal"] != "LONG":
        continue

    entry = work.iloc[i+1]["close"]
    exit_ = work.iloc[i+H]["close"]

    gross = (exit_ - entry) / entry

    # regime impact
    regime = work.iloc[i]["LARE"]

    slippage = 0.0003 + 0.3 * regime + 0.0005 * abs(work.iloc[i]["ddd_strength_h04c"])

    net = gross - slippage - 0.0004

    trades.append((regime, net))

trades_df = pd.DataFrame(trades, columns=["regime", "net"])

print("\n=== LARE EFFECT ===")
print(trades_df.groupby(pd.qcut(trades_df["regime"], 5, duplicates="drop"))["net"].mean())

print("\n=== CORRELATION ===")
print(trades_df["net"].corr(trades_df["regime"]))

=== LARE STATS ===
count    2789.000000
mean        0.030704
std         0.005125
min         0.019902
25%         0.027422
50%         0.029949
75%         0.033086
max         0.056907
Name: LARE, dtype: float64

=== LARE EFFECT ===
regime
(0.0189, 0.0267]    0.021135
(0.0267, 0.0289]    0.058637
(0.0289, 0.031]     0.032462
(0.031, 0.034]      0.055290
(0.034, 0.0569]     0.044089
Name: net, dtype: float64

=== CORRELATION ===
0.051439065199561464


In [36]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

H = 36

equity = 1.0
equity_curve = []
drawdowns = []

peak = 1.0

trades = []

for i in range(len(work) - H):

    if work.iloc[i]["signal"] != "LONG":
        continue

    entry_price = work.iloc[i+1]["close"]
    exit_price = work.iloc[i+H]["close"]

    strength = abs(work.iloc[i]["ddd_strength_h04c"])

    # volatility proxy
    vol = work["close"].pct_change().rolling(50).std().iloc[i]
    if np.isnan(vol):
        vol = 0.001

    # execution cost model
    slippage = 0.0003 + 0.4 * vol + 0.0007 * strength
    fee = 0.0004

    gross_ret = (exit_price - entry_price) / entry_price
    net_ret = gross_ret - slippage - fee

    # position sizing (fixed for now)
    size = 0.01  # 1% capital per trade

    pnl = equity * size * net_ret
    equity += pnl

    equity_curve.append(equity)

    peak = max(peak, equity)
    dd = (equity - peak) / peak
    drawdowns.append(dd)

    trades.append(net_ret)

trades = np.array(trades)
equity_curve = np.array(equity_curve)
drawdowns = np.array(drawdowns)

print("=== FINAL PAPER TRADING RESULTS ===")

print("Final Equity:", equity)
print("Total Return:", equity - 1)

print("\n=== RISK METRICS ===")
print("Max Drawdown:", drawdowns.min())
print("Mean Trade Return:", trades.mean())
print("Win Rate:", (trades > 0).mean())

print("\n=== DISTRIBUTION ===")
print("Std Dev:", trades.std())
print("Sharpe (approx):", trades.mean() / (trades.std() + 1e-9))

=== FINAL PAPER TRADING RESULTS ===
Final Equity: 3.799706281419506
Total Return: 2.799706281419506

=== RISK METRICS ===
Max Drawdown: -0.3661477299671816
Mean Trade Return: 0.04784388355440951
Win Rate: 0.5344273992151266

=== DISTRIBUTION ===
Std Dev: 0.20395398777155507
Sharpe (approx): 0.23458174974944235


In [37]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

# basic features
work["ret"] = work["close"].pct_change()
work["vol"] = work["ret"].rolling(50).std().fillna(0.001)

strength = work["ddd_strength_h04c"].abs()

# simple alpha quality score (no overengineering)
work["AQS"] = strength / (work["vol"] + 1e-6)

# quantile filtering
threshold = work["AQS"].quantile(0.7)

filtered = work[work["AQS"] >= threshold].copy()

print("=== FILTERING STATS ===")
print("Total signals:", len(work))
print("Filtered signals:", len(filtered))
print("Retention:", len(filtered)/len(work))

# backtest filtered set
H = 36

equity = 1.0
equity_curve = []
trades = []

for i in range(len(filtered)-H):

    entry = filtered.iloc[i+1]["close"]
    exit_ = filtered.iloc[i+H]["close"]

    gross = (exit_ - entry) / entry

    # simple cost
    cost = 0.0007 + 0.0004

    net = gross - cost

    equity *= (1 + 0.01 * net)

    equity_curve.append(equity)
    trades.append(net)

trades = np.array(trades)

print("\n=== FILTERED STRATEGY RESULTS ===")
print("Final Equity:", equity)
print("Mean Trade:", trades.mean())
print("Sharpe:", trades.mean() / (trades.std() + 1e-9))
print("Win Rate:", (trades > 0).mean())

=== FILTERING STATS ===
Total signals: 2839
Filtered signals: 852
Retention: 0.30010567101091934

=== FILTERED STRATEGY RESULTS ===
Final Equity: 5.712936916933196
Mean Trade: 0.21495941421419748
Sharpe: 0.44497664190518554
Win Rate: 0.5588235294117647


In [38]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

work["ret"] = work["close"].pct_change()
work["vol"] = work["ret"].rolling(50).std().fillna(0.001)

strength = work["ddd_strength_h04c"].abs()
work["AQS"] = strength / (work["vol"] + 1e-6)

threshold = work["AQS"].quantile(0.7)
filtered = work[work["AQS"] >= threshold].copy()

H = 36

equity = 1.0
equity_curve = []
trades = []

for i in range(len(filtered)-H):

    entry = filtered.iloc[i+1]["close"]
    exit_ = filtered.iloc[i+H]["close"]

    gross = (exit_ - entry) / entry

    aqs = filtered.iloc[i]["AQS"]
    vol = filtered.iloc[i]["vol"]

    # risk-aware sizing
    size = min(0.03, (aqs / (vol + 1e-6)) * 0.001)

    cost = 0.0007 + 0.0004

    net = gross - cost

    equity *= (1 + size * net)

    equity_curve.append(equity)
    trades.append(size * net)

trades = np.array(trades)

print("=== RISK-SIZED STRATEGY ===")
print("Final Equity:", equity)
print("Mean Trade:", trades.mean())
print("Sharpe:", trades.mean() / (trades.std() + 1e-9))
print("Max Exposure Approx:", max(equity_curve)/min(equity_curve))

=== RISK-SIZED STRATEGY ===
Final Equity: 40.804383306940075
Mean Trade: 0.004612440151793369
Sharpe: 0.4289815032547307
Max Exposure Approx: 40.46863642273096


In [39]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

# returns / volatility
work["ret"] = work["close"].pct_change()
work["vol"] = work["ret"].rolling(50).std().fillna(0.001)

# strength
work["strength"] = work["ddd_strength_h04c"].abs()

# alpha quality score
work["AQS"] = work["strength"] / (work["vol"] + 1e-6)

# ranks (CRITICAL CHANGE)
work["aqs_rank"] = work["AQS"].rank(pct=True)
work["vol_rank"] = work["vol"].rank(pct=True)

# filtering (same as before)
filtered = work[work["aqs_rank"] >= 0.7].copy()

H = 36

equity = 1.0
equity_curve = []
trades = []

for i in range(len(filtered) - H):

    entry = filtered.iloc[i+1]["close"]
    exit_ = filtered.iloc[i+H]["close"]

    gross = (exit_ - entry) / entry

    aqs_r = filtered.iloc[i]["aqs_rank"]
    vol_r = filtered.iloc[i]["vol_rank"]

    # bounded sizing function
    size = (aqs_r * (1 - vol_r)) * 0.02  # max ~2%

    size = np.clip(size, 0.001, 0.02)

    cost = 0.0007 + 0.0004

    net = gross - cost

    equity *= (1 + size * net)

    equity_curve.append(equity)
    trades.append(size * net)

trades = np.array(trades)

equity_curve = np.array(equity_curve)

drawdown = equity_curve / np.maximum.accumulate(equity_curve) - 1

print("=== RANK-BASED RISK ENGINE ===")
print("Final Equity:", equity)
print("Mean Trade:", trades.mean())
print("Sharpe:", trades.mean() / (trades.std() + 1e-9))
print("Max Drawdown:", drawdown.min())
print("Avg Size:", np.mean([ (filtered.iloc[i]["aqs_rank"] * (1 - filtered.iloc[i]["vol_rank"])) * 0.02 for i in range(len(filtered)-H) ]))

=== RANK-BASED RISK ENGINE ===
Final Equity: 7.277507037000486
Mean Trade: 0.0024554062362601966
Sharpe: 0.385230681402349
Max Drawdown: -0.28266352713432863
Avg Size: 0.0102954911871965


In [40]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

H = 36
trades = []

for i in range(len(work) - H):

    if work.iloc[i]["signal"] != "LONG":
        continue

    entry = work.iloc[i+1]["close"]
    exit_ = work.iloc[i+H]["close"]

    gross = (exit_ - entry) / entry

    vol = work["close"].pct_change().rolling(50).std().iloc[i]
    if np.isnan(vol):
        vol = 0.001

    cost = 0.0007 + 0.0004
    net = gross - cost

    trades.append({
        "net": net,
        "strength": work.iloc[i]["ddd_strength_h04c"],
        "vol": vol
    })

trades = pd.DataFrame(trades)

# 1. Sort by return
trades_sorted = trades.sort_values("net", ascending=False).reset_index(drop=True)

# cumulative contribution
trades_sorted["cum_pnl"] = trades_sorted["net"].cumsum()

total_pnl = trades_sorted["net"].sum()

print("=== PnL CONCENTRATION ===")
print("Top 5% contribution:",
      trades_sorted.head(int(len(trades_sorted)*0.05))["net"].sum() / total_pnl)

print("Top 10 trades contribution:",
      trades_sorted.head(10)["net"].sum() / total_pnl)

print("\n=== TAIL DEPENDENCE TEST ===")
print("Total PnL:", total_pnl)
print("PnL without top 5% trades:",
      trades_sorted.iloc[int(len(trades_sorted)*0.05):]["net"].sum())

print("\n=== QUANTILE STABILITY ===")
trades["bucket"] = pd.qcut(trades["strength"], 5, duplicates="drop")
print(trades.groupby("bucket")["net"].mean())

print("\n=== DISTRIBUTION SHAPE ===")
print(trades["net"].describe())
print("Skew:", trades["net"].skew())
print("Kurtosis:", trades["net"].kurt())

=== PnL CONCENTRATION ===
Top 5% contribution: 0.45414382086220556
Top 10 trades contribution: 0.04579541414643531

=== TAIL DEPENDENCE TEST ===
Total PnL: 165.75177484524193
PnL without top 5% trades: 90.47663050233172

=== QUANTILE STABILITY ===
bucket
(-0.0804, -0.013]       0.062836
(-0.013, -0.0104]       0.057581
(-0.0104, -0.00905]     0.052927
(-0.00905, -0.00815]    0.056820
(-0.00815, -0.00753]    0.065497
Name: net, dtype: float64

=== DISTRIBUTION SHAPE ===
count    2803.000000
mean        0.059134
std         0.203630
min        -0.504942
25%        -0.080109
50%         0.025284
75%         0.175869
max         0.883645
Name: net, dtype: float64
Skew: 0.5870331105997252
Kurtosis: 0.3475999385260127


In [41]:
import pandas as pd
import numpy as np

df = pd.read_parquet("data/signals/h04c_executable_signals.parquet")
work = df.reset_index(drop=True).copy()

H = 36

def simulate(pattern_filter):
    trades = []

    for i in range(len(work) - H):

        pattern = work.iloc[i]["pattern_h04c"]

        if pattern not in pattern_filter:
            continue

        entry = work.iloc[i+1]["close"]
        exit_ = work.iloc[i+H]["close"]

        gross = (exit_ - entry) / entry
        cost = 0.0011

        net = gross - cost

        trades.append(net)

    trades = np.array(trades)

    if len(trades) == 0:
        return None

    return {
        "n": len(trades),
        "mean": trades.mean(),
        "std": trades.std(),
        "sharpe": trades.mean() / (trades.std() + 1e-9),
        "win_rate": (trades > 0).mean(),
        "sum": trades.sum()
    }

# LONG side (DDD)
long_result = simulate(["DDD"])

# SHORT side (UUU)
short_result = simulate(["UUU"])

print("=== LONG (DDD) ===")
print(long_result)

print("\n=== SHORT (UUU) ===")
print(short_result)

# symmetry test
if long_result and short_result:
    print("\n=== SYMMETRY DIAGNOSTIC ===")
    print("mean ratio (LONG/SHORT):",
          long_result["mean"] / (abs(short_result["mean"]) + 1e-9))

=== LONG (DDD) ===
{'n': 2803, 'mean': np.float64(0.05913370490376094), 'std': np.float64(0.20359401520260134), 'sharpe': np.float64(0.2904491301203841), 'win_rate': np.float64(0.5658223332144131), 'sum': np.float64(165.75177484524193)}

=== SHORT (UUU) ===
None
